In [ ]:
from transformers import T5ForConditionalGeneration,T5Tokenizer

In [ ]:
qa_corpus =[
    "Q: What are the symptoms of diabetes? A: common symptoms include increased thirst, frequent urination,and unexplained weight loss",
    "Q: How is hypertension treated? A: Treatments includes lifestyle changes,medications,and regularmonitoring of blood pressure.",
]

summarization_corpus = [
    "Text: diabetes is a chronic condition characterized by high high levels of sugar in the blood,Summary: Diabetes is a chronic condition with high blood sugar levels.",
    "Text: Hypertension, is a condition in which the force of the blood against the artery walls is too high.Summary: Hypertension is a condition with high blood pressure",
]

In [ ]:
from torch.utils.data import Dataset

class TSDataset(Dataset):
  def __init__(self,texts,tokenizer,block_size=128):
    self.examples = []
    for text in texts:
      self.examples.append(tokenizer(text,trunction=True,padding='max_length',max_length = block_size,return_tensors="pt"))

  def __len__(self):
    return len(self.examples)

  def __getitem__(self,i):
    return {key:val.squeeze() for key,val in self.examples[i].items()}

In [ ]:
import torch
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

qa_dataset = TSDataset(qa_corpus,tokenizer)
summarization_dataset = TSDataset(summarization_corpus,tokenizer)

combined_dataset = torch.utils.data.ConcatDataset([qa_dataset,summarization_dataset])

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Keyword arguments {'trunction': True} not recognized.
Keyword arguments {'trunction': True} not recognized.
Keyword arguments {'trunction': True} not recognized.
Keyword arguments {'trunction': True} not recognized.


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel, DataCollatorForLanguageModeling

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.generation_config.pad_token_id = tokenizer.eos_token_id

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = './results',
    overwrite_output_dir=True,
    num_train_epochs = 3,
    per_device_train_batch_size =4,
    save_steps =10_000,
    save_total_limit =2,)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model =model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = combined_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jas1420047 (jas1420047-mn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


TrainOutput(global_step=3, training_loss=5.560895919799805, metrics={'train_runtime': 114.9891, 'train_samples_per_second': 0.104, 'train_steps_per_second': 0.026, 'total_flos': 783876096000.0, 'train_loss': 5.560895919799805, 'epoch': 3.0})

In [ ]:
def generate_response_multi_task(query,model,tokenizer,task_prefix=""):
  input_text = task_prefix+query
  input_ids = tokenizer.encode(input_text,return_tensors = 'pt')
  output = model.generate(input_ids,max_length=200,num_return_sequences=1)
  response= tokenizer.decode(output[0],skip_special_tokens=True)
  return response

qa_query = "Q: What are the symptoms of diabetes?"
summarization_query = "Text: Diabetes is a chronic condition characterized by high levels of sugar or glucose in the blood"

qa_response = generate_response_multi_task(qa_query,model,tokenizer,task_prefix="A: ")
summarization_response = generate_response_multi_task(summarization_query,model,tokenizer,task_prefix= "Summary: ")

print("QA response: ",qa_response)
print("Summarization response: ",summarization_response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


QA response:  A: Q: What are the symptoms of diabetes? A: Diabetes is a condition that is caused by the body's own internal organs to produce insulin. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition that is caused by the body's own internal organs. It is a condition
Summarization response:  Summary: Text: Diabetes is a chronic condition characteriz